# Pedro Reasoning Benchmarks - Colab

This notebook runs the reasoning benchmark set from the project table:

- ARC Challenge (0-shot)
- GPQA (0-shot)
- HellaSwag (0-shot)

Models:

- Llama 3.2 1B
- Llama 3.2 3B
- Phi-3.5-mini
- Gemma 2 2B

The notebook uses the repository's `scripts/run_benchmarks.py` runner, then builds a summary CSV and chart from the saved JSON outputs.

## 1. Install and Open the Repo

In [ ]:
# Colab setup. If you opened this notebook from inside the repo, this cell will reuse it.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/PedroPovedaQ/LLaMA-3.2-Lightweight-Text-and-Multimodal-Models.git"
REPO_DIR = Path("/content/LLaMA-3.2-Lightweight-Text-and-Multimodal-Models")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=False)
else:
    def find_project_root(start=None):
        start = Path(start or os.getcwd()).resolve()
        for path in [start, *start.parents]:
            if (path / "scripts" / "run_benchmarks.py").exists():
                return path
        raise RuntimeError("Run this notebook inside the repository or in Colab.")
    os.chdir(find_project_root())

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Install runtime dependencies. On Colab, restart the runtime if pip asks for it.
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate", "sentencepiece", "protobuf"], check=True)

## 2. Runtime, Drive, and Hugging Face Login

In [ ]:
import platform
import shutil

try:
    import torch
except Exception as exc:
    torch = None
    print(f"PyTorch import failed: {exc}")

print(f"Python: {platform.python_version()}")
print(f"Free disk: {shutil.disk_usage(PROJECT_ROOT).free / (1024**3):.1f} GB")

if torch is not None:
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

In [ ]:
# Persist result files in Google Drive when running in Colab.
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = Path("/content/drive/MyDrive/pedro_reasoning_results")
else:
    RESULTS_DIR = PROJECT_ROOT / "results" / "pedro_reasoning_colab"

RAW_DIR = RESULTS_DIR / "raw"
REPORT_DIR = RESULTS_DIR / "reports"
RAW_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results dir: {RESULTS_DIR}")

In [ ]:
# Hugging Face auth is required for Llama/Gemma licenses and GPQA access.
# In Colab, you can store HF_TOKEN in Secrets. Otherwise this falls back to a prompt.
from getpass import getpass

DO_HF_LOGIN = True

if DO_HF_LOGIN:
    from huggingface_hub import login
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        token = getpass("Hugging Face token: ")
    login(token=token)
else:
    print("Skipping HF login.")

## 3. Benchmark Configuration

In [ ]:
MODELS = {
    "llama-3.2-1b": {
        "label": "Llama 3.2 1B",
        "hf_id": "meta-llama/Llama-3.2-1B-Instruct",
        "color": "#4C78A8",
    },
    "llama-3.2-3b": {
        "label": "Llama 3.2 3B",
        "hf_id": "meta-llama/Llama-3.2-3B-Instruct",
        "color": "#C44E52",
    },
    "phi-3.5-mini": {
        "label": "Phi-3.5-mini",
        "hf_id": "microsoft/Phi-3.5-mini-instruct",
        "color": "#8CBF40",
    },
    "gemma-2-2b": {
        "label": "Gemma 2 2B",
        "hf_id": "google/gemma-2-2b-it",
        "color": "#8064A2",
    },
}

BENCHMARKS = ["arc", "gpqa", "hellaswag"]
BENCHMARK_LABELS = {
    "arc": "ARC Challenge",
    "gpqa": "GPQA",
    "hellaswag": "HellaSwag",
}

# Start with LIMIT = 5 for a smoke test. Use a larger limit for final charts.
LIMIT = 100
PRECISION = "fp16"
DEVICE = "auto"
SAVE_PREDICTIONS = False

print("Models:")
for key, spec in MODELS.items():
    print(f"  {key}: {spec['hf_id']}")
print(f"Benchmarks: {BENCHMARKS}")
print(f"Limit per benchmark: {LIMIT}")

## 4. Run Benchmarks

In [ ]:
import json
import time

from generate_benchmark_report import generate_benchmark_report

RUN_BENCHMARKS = True

def run_one_model(model_key, spec):
    output_json = RAW_DIR / f"{model_key}__reasoning_limit{LIMIT}.json"
    output_prefix = REPORT_DIR / f"{model_key}__reasoning_limit{LIMIT}"
    cmd = [
        sys.executable,
        "scripts/run_benchmarks.py",
        "--model-id", spec["hf_id"],
        "--benchmarks", *BENCHMARKS,
        "--limit", str(LIMIT),
        "--precision", PRECISION,
        "--device", DEVICE,
        "--output", str(output_json),
        "--no-report",
    ]
    if SAVE_PREDICTIONS:
        cmd.append("--save-predictions")

    print("\n[RUN]", spec["label"])
    print(" ".join(cmd))
    started = time.time()
    subprocess.run(cmd, check=True)
    elapsed = time.time() - started

    with output_json.open("r", encoding="utf-8") as f:
        data = json.load(f)
    artifacts = generate_benchmark_report(data, output_prefix)
    print(f"[DONE] {spec['label']} in {elapsed / 60:.1f} min")
    print(f"JSON: {output_json}")
    print(f"PDF:  {artifacts['report_pdf']}")
    return output_json

if RUN_BENCHMARKS:
    completed = []
    for model_key, spec in MODELS.items():
        completed.append(run_one_model(model_key, spec))
else:
    completed = sorted(RAW_DIR.glob(f"*__reasoning_limit{LIMIT}.json"))
    print("Skipping runs. Found existing JSON files:")
    for path in completed:
        print(" ", path)

## 5. Build Summary Table

In [ ]:
import pandas as pd

rows = []
for model_key, spec in MODELS.items():
    path = RAW_DIR / f"{model_key}__reasoning_limit{LIMIT}.json"
    if not path.exists():
        for bench in BENCHMARKS:
            rows.append({"model": spec["label"], "benchmark": bench, "accuracy": None, "n": None, "duration_sec": None})
        continue

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data["benchmarks"]:
        rows.append({
            "model": spec["label"],
            "model_key": model_key,
            "benchmark": item["name"],
            "benchmark_label": BENCHMARK_LABELS[item["name"]],
            "accuracy": item["accuracy"],
            "n": item["num_examples"],
            "correct": item["correct"],
            "duration_sec": item["duration_sec"],
        })

df_long = pd.DataFrame(rows)
df_pivot = df_long.pivot_table(index="model", columns="benchmark", values="accuracy").reindex([spec["label"] for spec in MODELS.values()])
df_pivot = df_pivot[BENCHMARKS]
df_pivot["macro_avg"] = df_pivot.mean(axis=1)

summary_csv = REPORT_DIR / f"pedro_reasoning_summary_limit{LIMIT}.csv"
df_pivot.to_csv(summary_csv)

print("Long format:")
print(df_long.to_string(index=False))
print("\nPivot accuracy table:")
print(df_pivot.to_string(float_format="{:.3f}".format))
print(f"\nSaved: {summary_csv}")

## 6. Generate Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

labels = [BENCHMARK_LABELS[b] for b in BENCHMARKS]
x = np.arange(len(labels))
width = 0.18
offsets = np.linspace(-1.5 * width, 1.5 * width, len(MODELS))

fig, ax = plt.subplots(figsize=(12, 6.5))
for offset, (model_key, spec) in zip(offsets, MODELS.items()):
    values = [df_pivot.loc[spec["label"], bench] for bench in BENCHMARKS]
    bars = ax.bar(x + offset, values, width, label=spec["label"], color=spec["color"])
    ax.bar_label(bars, labels=[f"{100 * v:.0f}" if pd.notna(v) else "n/a" for v in values], padding=3, fontsize=8)

ax.set_title("Reasoning Benchmarks")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(ncol=4, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.13))
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

chart_path = REPORT_DIR / f"pedro_reasoning_accuracy_limit{LIMIT}.png"
fig.savefig(chart_path, dpi=180)
plt.close(fig)

print(f"Saved chart: {chart_path}")
display(Image(filename=str(chart_path)))

## 7. Files Produced

In [ ]:
print("Raw JSON files:")
for path in sorted(RAW_DIR.glob("*.json")):
    print(" ", path)

print("\nReport files:")
for path in sorted(REPORT_DIR.glob("*")):
    print(" ", path)